# 08 - שוויון סוציו-אקונומי ופרדוקס סימפסון

מחברת זו בוחנת האם רשת האוטובוסים והרכבות בישראל מספקת **רמת שירות גבוהה יותר לשכונות אמידות**. כל תחנה משויכת לאזור סטטיסטי סוציו-אקונומי של הלשכה המרכזית לסטטיסטיקה (הלמ"ס) לשנת 2021, השירות מצטבר לרמת השכונה (האזור הסטטיסטי), והאשכול הסוציו-אקונומי (1 = החלש ביותר, 10 = החזק ביותר) מתואם עם השירות לנפש.

התוצאה המרכזית היא **פרדוקס סימפסון (Simpson's paradox)**: במצרף ארצי הקשר חלש ושלילי במקצת, אך *בתוך* מטרופולין בודד (המיוצג כאן על ידי קהילת Louvain של גרף הנסיעות) הקשר לרוב חזק - והוא מצביע על **כיוונים הפוכים במטרופולינים שונים**, ולכן ההשפעות המקומיות מתקזזות בעת המצרף.

**שאלת המחקר.** האם המעמד הסוציו-אקונומי מנבא את רמת השירות התחבורתי לנפש בישראל, והאם התשובה ברמה הארצית זהה לתשובה בתוך עיר או מטרופולין בודדים?

**קלט**
- `outputs/nb/<earlier stage>/tables/stop_metrics.csv` - מדדים ברמת התחנה משלב בניית הגרף / מדדי המרכזיות (קואורדינטות התחנה, `stop_use_count`, `degree` של סמיכות נסיעות, קהילת Louvain, betweenness, דגל articulation point). קיימת נפילה לאחור אל הקובץ הקנוני של המאגר `outputs/tables/stop_metrics.csv`.
- אזורים סטטיסטיים סוציו-אקונומיים של הלמ"ס 2021, המורדים בזמן ריצה משירות ArcGIS REST (**תלות חיצונית - ראו האזהרה להלן**).

**פלט** (הכול תחת `outputs/nb/08_socioeconomic_equity/`)
- `tables/stops_with_socioeconomic.csv`, `tables/socioeconomic_neighborhood_access.csv`
- `tables/socioeconomic_cluster_summary.csv`, `tables/community_socioeconomic_summary.csv`
- `tables/socioeconomic_national_correlations.csv`, `tables/socioeconomic_within_cluster_correlation.csv`
- `tables/trend_fit_sensitivity.csv`, `tables/socioeconomic_join_quality.json`, `tables/socioeconomic_summary.json`
- `data/cbs_socioeconomic_areas_2021.geojson` (הורדה שמורה במטמון, כך שהרצות חוזרות אינן דורשות רשת)
- `figures/socioeconomic_access_by_cluster.png`, `figures/socioeconomic_cluster_average_flat.png`, `figures/socioeconomic_cities_no_rule.png`, `figures/socioeconomic_within_cluster_correlation.png`, `figures/socioeconomic_within_cluster_examples.png`, `figures/socioeconomic_simpson_paradox.png`

**זמן ריצה.** מספר דקות מקצה לקצה. שני השלבים היקרים הם הורדת הפוליגונים של הלמ"ס (כ-2,900 פוליגונים, נשמרים במטמון לאחר ההרצה הראשונה) ושיוך point-in-polygon של כ-30,000 תחנות.

**אזהרת מינוח.** המילה *cluster* משמשת במקורות לשני דברים שונים, ולכן מחברת זו מקפידה על ההבחנה:
- **אשכול סוציו-אקונומי (socioeconomic cluster)** = מדד הלמ"ס 1-10 של אזור סטטיסטי;
- **קהילה (community)** = קהילת Louvain של גרף התחבורה הציבורית, המשמשת כאן כפרוקסי מבוסס-נתונים לעיר / מטרופולין.

שמות קובצי האיורים שפורסמו שומרים על הניסוח המקורי (`within_cluster`) כדי שיישארו ברי-השוואה לדוח, אך תוויות הצירים מציינות *community*.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## תיקיות השלב וקבועים ניתנים לכוונון

כל מה שמחברת זו כותבת נמצא בתיקיית השלב הייעודית שלה, כך שלא ניתן לדרוס את התוצאות המצוטטות בדוח שב-`outputs/tables`, `outputs/figures` או `outputs/rail`.

הקבועים שלהלן רוכזו כאן משום שהם הידיות המשנות את התוצאה. אלה שראוי להכיר:

- `NEAREST_JOIN_MAX_DISTANCE_M` - תחנה שאינה נופלת בתוך אף פוליגון של הלמ"ס (כבישים כפריים, מחלפים, אזורי תעשייה) משויכת לפוליגון הקרוב ביותר עד למרחק זה. מעבר לו התחנה נותרת ללא שיוך, במקום להיכפות לשכונה שאינה שייכת אליה.
- `MIN_NEIGHBORHOODS` / `MIN_DISTINCT_SOCIO_LEVELS` - מתאם בתוך קהילה מחושב רק כאשר הקהילה מכילה מספיק שכונות, הפרוסות על פני מספיק רמות סוציו-אקונומיות שונות, כדי שמתאם הדירוגים יישא מידע.
- `DISPLAY_TRIM_PCT` - משמש **אך ורק** לחיתוך תצוגת ציר ה-y בתרשימי הפיזור. הוא לעולם אינו משמש להתאמת קו או לחישוב סטטיסטי (ראו ההערה על באג הגזימה בהמשך).
- `CITIES_IN_NO_RULE_FIGURE` - נותר `None` כך שהערים להמחשה נבחרות מתוך הנתונים ולא מקובעות בקוד.

In [ ]:
STAGE = OUT / '08_socioeconomic_equity'
(STAGE / 'tables').mkdir(parents=True, exist_ok=True)
(STAGE / 'figures').mkdir(parents=True, exist_ok=True)
(STAGE / 'data').mkdir(parents=True, exist_ok=True)

# --- External data source: CBS 2021 socioeconomic statistical areas (ArcGIS REST) ---
CBS_LAYER_URL = ('https://services2.arcgis.com/xMRYm7cNgdR5RN6F/arcgis/rest/services/'
                 'SOEC_Stat11_2021/FeatureServer/27')
CBS_PAGE_SIZE = 2000                      # features per request (the service caps a page at 2000)
CBS_CACHE = STAGE / 'data' / 'cbs_socioeconomic_areas_2021.geojson'

# Spatial join fallback distance, in metres (measured in EPSG:3857).
NEAREST_JOIN_MAX_DISTANCE_M = 3000

# A stop counts as critical if it is an articulation point or sits in the top
# decile of betweenness - the same rule the rest of the project uses.
CRITICAL_BETWEENNESS_QUANTILE = 0.90

# Within-community test thresholds.
MIN_NEIGHBORHOODS = 12
MIN_DISTINCT_SOCIO_LEVELS = 3
ALPHA = 0.05

# Figure-only thresholds. The saved tables keep every community that passes the
# thresholds above; the figures show a readable subset of them.
FIG_MIN_NEIGHBORHOODS = 20
TOP_COMMUNITIES_FOR_FIGURE = 15
EXAMPLE_MIN_NEIGHBORHOODS = 40
DISPLAY_TRIM_PCT = 90        # y-axis VIEW clipping only - never used for fitting or statistics

# Illustrative cities for the bar chart. None -> derive from the data.
CITIES_IN_NO_RULE_FIGURE = None
N_CITIES_NO_RULE = 6

print('Stage folder :', STAGE)
print('CBS cache    :', CBS_CACHE)

## תוויות בעברית ב-matplotlib

שמות היישובים מגיעים משכבת הלמ"ס בעברית, והם מופיעים בתוויות הצירים ובמקראות האיורים. matplotlib אינה מריצה את האלגוריתם הדו-כיווני (bidirectional) של Unicode, ולכן מחרוזות בעברית מוצגות הפוכות. התיקון שלהלן כותב מחדש את הטקסט לסדר תצוגה פעם אחת, לפני שמצויר איור כלשהו. טקסט לטיני עובר ללא שינוי, ולכן כותרות צירים באנגלית אינן מושפעות.

In [ ]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()

## ספריות

`geopandas` (יחד עם `shapely` ו-`pyproj`) מבצעת את שיוכי point-in-polygon ו-nearest-polygon; `requests` מדפדפת בעמודי שירות ה-ArcGIS; `scipy` מספקת את מתאם הדירוגים Spearman. `seaborn` משמשת אך ורק לערכת הנושא של התרשימים, ו-`install_hebrew()` נקראת שוב לאחר מכן משום ש-`sns.set_theme` מאפסת את משפחת הגופנים.

In [ ]:
_ensure('geopandas', 'requests', 'scipy', 'seaborn')

import json
from urllib.parse import urlencode

import numpy as np
import pandas as pd
import geopandas as gpd
import requests
import seaborn as sns
from scipy.stats import spearmanr

sns.set_theme(style='whitegrid', font_scale=1.0)
install_hebrew()   # sns.set_theme resets the font family; re-apply the Hebrew-capable fonts

print('pandas', pd.__version__, '| geopandas', gpd.__version__)

## מוסכמת הסימן (יש לקרוא זאת לפני התבוננות בכל מספר)

ביקורת על הגרסה הקודמת העלתה כי טקסט המצגת ציטט את ערך ה-Spearman rho **הגולמי**, בעוד שהתרשים שרטט `equity_bias = -rho`, ולכן אותה עיר דווחה כ-`+0.48` במקום אחד וכ-`-0.48` במקום אחר.

**מחברת זו משתמשת ב-Spearman rho הגולמי בכל מקום - בטבלאות, בתוויות הצירים ובטקסט.** שום סימן אינו מתהפך. מכיוון שהאשכול הסוציו-אקונומי נע בין 1 (החלש ביותר) ל-10 (החזק ביותר):

| rho | משמעות |
| --- | --- |
| **rho < 0** | השירות לנפש **גבוה יותר בשכונות החלשות** |
| **rho > 0** | השירות לנפש **גבוה יותר בשכונות החזקות** |

הצבע משמש רק כרמז חוזר לאותו כיוון עצמו (ירוק = rho שלילי, אדום = rho חיובי); המספר המודפס על כל עמודה הוא ה-rho הגולמי.

### מוסכמה שנייה: התאמות וסטטיסטיקות חייבות להתיישב זו עם זו

הגרסה הקודמת שרטטה קווי מגמה שהותאמו על נתונים שנגזמו לפי אחוזונים (**trimmed**), בעוד שציטטה Spearman **לא גזום** כסטטיסטיקה המרכזית, כך שהקו והמספר תיארו מערכי נתונים שונים. כאן כל קו מגמה הוא התאמת ריבועים פחותים (OLS) על **בדיוק אותן שורות לא גזומות** הנכנסות לחישוב ה-Spearman המדווח. גזימה לפי אחוזונים שורדת רק כהגדרת *תצוגה* של ציר ה-y, ו-`tables/trend_fit_sensitivity.csv` מתעד עד כמה השיפוע היה משתנה לו הגזימה הישנה הייתה מיושמת, כך שהבחירה ניתנת לביקורת ואינה מוסתרת.

In [ ]:
RHO_AXIS_LABEL = 'Spearman rho  (socioeconomic cluster  vs  service per 1,000 residents)'
RHO_NEG_MEANING = 'rho < 0  ->  more service per capita in the WEAKER neighbourhoods'
RHO_POS_MEANING = 'rho > 0  ->  more service per capita in the STRONGER neighbourhoods'

NEG_COLOR = '#1e8449'   # green: negative rho
POS_COLOR = '#c0392b'   # red:   positive rho

def rho_color(rho):
    """Colour encodes the SIGN OF THE RAW RHO - no sign flipping anywhere."""
    return NEG_COLOR if rho < 0 else POS_COLOR

def describe_rho(rho):
    if rho is None or (isinstance(rho, float) and np.isnan(rho)):
        return 'undefined'
    return 'weaker neighbourhoods favoured' if rho < 0 else 'stronger neighbourhoods favoured'

def spearman(x, y, min_n=8):
    """Spearman rank correlation on the raw (untrimmed) overlapping rows.

    Returns (rho, p_value, n). Returns NaNs when the sample is too small or one
    of the variables is constant, so the caller never sees a meaningless 1.0.
    """
    sample = pd.DataFrame({'x': pd.to_numeric(pd.Series(x).reset_index(drop=True), errors='coerce'),
                           'y': pd.to_numeric(pd.Series(y).reset_index(drop=True), errors='coerce')}).dropna()
    if len(sample) < min_n or sample['x'].nunique() < 2 or sample['y'].nunique() < 2:
        return np.nan, np.nan, len(sample)
    rho, p_value = spearmanr(sample['x'], sample['y'])
    return float(rho), float(p_value), int(len(sample))

def ols_slope(x, y):
    """Least-squares slope/intercept on the untrimmed overlapping rows."""
    sample = pd.DataFrame({'x': pd.to_numeric(pd.Series(x).reset_index(drop=True), errors='coerce'),
                           'y': pd.to_numeric(pd.Series(y).reset_index(drop=True), errors='coerce')}).dropna()
    if len(sample) < 2 or sample['x'].nunique() < 2:
        return None
    return np.polyfit(sample['x'], sample['y'], 1)

def per_1000(numerator, denominator):
    numerator = pd.to_numeric(numerator, errors='coerce')
    denominator = pd.to_numeric(denominator, errors='coerce')
    return np.where(denominator > 0, numerator / denominator * 1000, np.nan)

def mode_or_na(series):
    values = series.dropna()
    return values.mode().iloc[0] if len(values) else np.nan

def save_fig(fig, name):
    path = STAGE / 'figures' / name
    fig.savefig(path, dpi=160, bbox_inches='tight')
    plt.close(fig)
    print('saved', path.name)
    return path

print(RHO_NEG_MEANING)
print(RHO_POS_MEANING)

## טעינת מדדי הגרף ברמת התחנה

שלב זה אינו בונה מחדש את הגרף; הוא צורך את טבלת התחנות שהופקה בשלב בניית הגרף / מדדי המרכזיות. הטוען מחפש תחילה את `outputs/nb/*/tables/stop_metrics.csv` (כלומר מחברת מוקדמת יותר בסדרה זו) ונופל לאחור אל הקובץ הקנוני של המאגר `outputs/tables/stop_metrics.csv`, המצורף לגרסת המקור. אם אף אחד מהם אינו קיים, נזרקת שגיאה ברורה ובת-פעולה במקום ניתוח שקט של לא כלום.

**באג שתוקן כאן.** הקובץ שפורסם `stops_with_socioeconomic.csv` נשא עמודת `degree` שירשה מ**גרף קרבה מרחבית של 500 מ' שהוצא משימוש**, וטבלת השכונות צברה אותה כ-`avg_proximity_degree`. דרגת קרבה מודדת עד כמה צפופה סביבת 500 המטרים של תחנה - זהו למעשה מדד צפיפות, לא מדד שירות - ולכן אין לה מקום בניתוח מבנה הרשת. מחברת זו לוקחת את `degree` מ**גרף סמיכות הנסיעות** (שתי תחנות סמוכות כאשר נסיעה מתוכננת משרתת אותן ברצף) ומכנה אותה `trip_graph_degree` בכל מקום, כך שהעמודה המיושנת לא תוכל לחזור בשוגג.

In [ ]:
def find_stop_metrics():
    """Prefer an earlier notebook stage; fall back to the committed pipeline table."""
    candidates = sorted(OUT.glob('*/tables/stop_metrics.csv'))
    canonical = REPO / 'outputs' / 'tables' / 'stop_metrics.csv'
    if canonical.exists():
        candidates.append(canonical)
    if not candidates:
        raise FileNotFoundError(
            'No stop_metrics.csv found under ' + str(OUT) + '/*/tables/ and no '
            + str(canonical) + '. Run the graph-construction / centrality notebook '
            '(e.g. 02_graph_construction) first.')
    return candidates[0]

STOP_METRICS_PATH = find_stop_metrics()
print('Reading per-stop metrics from:', STOP_METRICS_PATH)

metrics = pd.read_csv(STOP_METRICS_PATH, encoding='utf-8-sig', low_memory=False)
metrics.columns = [str(c).strip().lstrip('\ufeff') for c in metrics.columns]

# The Louvain community label is produced by notebook 09, not by the centrality
# stage, so stop_metrics.csv does not carry it. Merge it in when it is absent.
_COMM_NAMES = ['community_id', 'community_louvain', 'community']
if not any(c in metrics.columns for c in _COMM_NAMES):
    _comm_files = sorted(OUT.glob('09*/tables/community_assignments.csv'))
    if not _comm_files:
        raise FileNotFoundError(
            'No community_assignments.csv under ' + str(OUT) + '/09*/tables/. '
            'Run 09_community_detection.ipynb first - this notebook needs Louvain '
            'communities to compute the within-cluster correlations.')
    _comm = pd.read_csv(_comm_files[0], encoding='utf-8-sig', low_memory=False)
    _comm.columns = [str(c).strip() for c in _comm.columns]
    _cc = next((c for c in _COMM_NAMES if c in _comm.columns), None)
    if _cc is None:
        raise KeyError(str(_comm_files[0]) + ' has no community column; found: '
                       + str(list(_comm.columns)))
    metrics['stop_id'] = metrics['stop_id'].astype(str)
    _comm['stop_id'] = _comm['stop_id'].astype(str)
    metrics = metrics.merge(_comm[['stop_id', _cc]], on='stop_id', how='left')
    print('Merged Louvain communities from:', _comm_files[0])
    print('  stops with a community:', int(metrics[_cc].notna().sum()))

def pick_column(frame, names, what, required=True):
    for name in names:
        if name in frame.columns:
            return name
    if required:
        raise KeyError(str(STOP_METRICS_PATH) + ' has none of ' + str(names)
                       + ' (needed for ' + what + ').')
    return None

lat_col = pick_column(metrics, ['stop_lat', 'lat'], 'stop latitude')
lon_col = pick_column(metrics, ['stop_lon', 'lon'], 'stop longitude')
use_col = pick_column(metrics, ['stop_use_count'], 'scheduled stop calls')
deg_col = pick_column(metrics, ['degree'], 'trip-adjacency degree')
com_col = pick_column(metrics, ['community_id', 'community_louvain', 'community'], 'Louvain community')
btw_col = pick_column(metrics, ['approx_betweenness', 'betweenness'], 'betweenness', required=False)

stops = pd.DataFrame({
    'stop_id': metrics['stop_id'].astype(str),
    'stop_name': metrics['stop_name'].astype(str) if 'stop_name' in metrics.columns else '',
    'lat': pd.to_numeric(metrics[lat_col], errors='coerce'),
    'lon': pd.to_numeric(metrics[lon_col], errors='coerce'),
    'stop_use_count': pd.to_numeric(metrics[use_col], errors='coerce').fillna(0.0),
    # NOTE: trip-adjacency degree, NOT the retired 500 m proximity degree.
    'trip_graph_degree': pd.to_numeric(metrics[deg_col], errors='coerce').fillna(0.0),
    'community': pd.to_numeric(metrics[com_col], errors='coerce'),
})
if 'weighted_degree' in metrics.columns:
    stops['trip_graph_weighted_degree'] = pd.to_numeric(metrics['weighted_degree'], errors='coerce').fillna(0.0)
if btw_col is not None:
    stops['betweenness'] = pd.to_numeric(metrics[btw_col], errors='coerce').fillna(0.0)
if 'is_articulation_point' in metrics.columns:
    stops['is_articulation_point'] = (metrics['is_articulation_point'].astype(str)
                                      .str.strip().str.lower().isin(['true', '1', 'yes']))

# Louvain marks unassigned nodes with -1 in some exports; that is missing, not a community.
stops.loc[stops['community'] < 0, 'community'] = np.nan
stops = stops.dropna(subset=['lat', 'lon']).reset_index(drop=True)

print('stops with coordinates      :', len(stops))
print('community coverage          : %.1f%% of stops, %d communities'
      % (100 * stops['community'].notna().mean(), stops['community'].dropna().nunique()))
print('columns kept                :', list(stops.columns))
assert 'proximity' not in ' '.join(stops.columns), 'A proximity-graph column leaked in'

## תוויות אזור ומטרופולין

טבלאות השכונות והקהילות נושאות אזור דומיננטי ומטרופולין דומיננטי, המאפשרים לקרוא את התוצאות באופן ברור ("קהילה זו היא אזור חיפה"). מדובר בתוויות גיאוגרפיות גסות הנגזרות מקואורדינטות התחנות - אותו כלל שבו השתמש שלב הכנת הנתונים של הפרויקט: תחילה תיבה תוחמת (bounding box) של ירושלים, לאחר מכן רצועות קו רוחב לצפון / מרכז / דרום, ורדיוס סביב כל אחד מארבעת מרכזי המטרופולין. הן תיאוריות בלבד ואינן משמשות אף פעם כמשתנה בניתוח.

In [ ]:
METRO_CENTERS = {
    'תל אביב': (32.0853, 34.7818, 30),
    'חיפה': (32.7940, 34.9896, 25),
    'ירושלים': (31.7683, 35.2137, 20),
    'באר שבע': (31.2518, 34.7913, 25),
}

def haversine_km(lat1, lon1, lat2, lon2):
    radius = 6371.0
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlam / 2) ** 2
    return 2 * radius * np.arcsin(np.sqrt(a))

def assign_region(lat, lon):
    if 31.70 <= lat <= 31.90 and 34.95 <= lon <= 35.30:
        return 'ירושלים'
    if lat > 32.50:
        return 'צפון'
    if lat >= 31.55:
        return 'מרכז'
    return 'דרום'

def assign_metro(lat, lon):
    for city, (clat, clon, radius) in METRO_CENTERS.items():
        if haversine_km(lat, lon, clat, clon) <= radius:
            return city
    return 'פריפריה'

stops['region'] = [assign_region(a, b) for a, b in zip(stops['lat'], stops['lon'])]
stops['metro'] = [assign_metro(a, b) for a, b in zip(stops['lat'], stops['lon'])]

print(stops['region'].value_counts().to_string())
print()
print(stops['metro'].value_counts().to_string())

## דגל קריטיות

טבלאות השוויון מדווחות מהו חלקן של התחנות ה*קריטיות* בשכונה, לפי ההגדרה של הפרויקט: תחנה היא קריטית אם היא נקודת חיתוך (articulation point) - הסרתה מנתקת חלק מהרשת - **או** אם ה-betweenness שלה נמצא בעשירון העליון. אם בטבלת המדדים חסרות עמודות אלה, הדגל נקבע ל-`False` והחלק היחסי פשוט הופך ללא-אינפורמטיבי במקום שגוי.

In [ ]:
if 'betweenness' in stops.columns and 'is_articulation_point' in stops.columns:
    threshold = float(stops['betweenness'].quantile(CRITICAL_BETWEENNESS_QUANTILE))
    stops['is_critical'] = stops['is_articulation_point'] | (stops['betweenness'] >= threshold)
    print('betweenness p%d threshold : %.6g' % (CRITICAL_BETWEENNESS_QUANTILE * 100, threshold))
    print('articulation points       :', int(stops['is_articulation_point'].sum()))
    print('critical stops            : %d (%.1f%%)'
          % (int(stops['is_critical'].sum()), 100 * stops['is_critical'].mean()))
else:
    stops['is_critical'] = False
    print('WARNING: no betweenness / articulation-point columns in the metrics table;')
    print('         critical_stop_share will be 0 everywhere and should be ignored.')

## תלות בנתונים חיצוניים: שכבת הלמ"ס הסוציו-אקונומית 2021

> **סיכון תלות בנתונים.** המדד הסוציו-אקונומי **אינו מצורף למאגר זה**. הוא מורד בזמן ריצה משירות feature ציבורי מסוג ArcGIS REST המתארח עבור הלשכה המרכזית לסטטיסטיקה (`services2.arcgis.com/.../SOEC_Stat11_2021/FeatureServer/27`, כ-2,900 פוליגונים של אזורים סטטיסטיים). אם השירות ישנה את שמו, יוצא משימוש או ייחסם בחומת אש, מחברת זו לא תוכל לשחזר את קלטיה ואין תחליף לא-מקוון במאגר. לפיכך ההרצה המוצלחת הראשונה שומרת במטמון את ההורדה הגולמית אל `outputs/nb/08_socioeconomic_equity/data/cbs_socioeconomic_areas_2021.geojson`; כל הרצה מאוחרת יותר קוראת מהמטמון ואינה זקוקה לרשת כלל. **שמרו את הקובץ השמור הזה** - הוא העותק הלא-מקוון היחיד.
>
> השכבה כבר השתנתה פעם אחת: השדות `CLUSTER_2021` ו-`INDEX_VALUE_2021` היו מאוכלסים בעת הרצת הניתוח המקורי, אך כעת הם מוגשים כ-`NULL` לחלוטין. לפיכך האשכול הסוציו-אקונומי נקרא מתוך `eshkol_mad` (קיים עבור כל הפוליגונים), וערך המדד הרציף עשוי לחזור ריק - המחברת בודקת ומדווחת על כך במקום להניח.

השירות מגביל תגובה ל-2,000 ישויות, ולכן ההורדה מדפדפת בשכבה באמצעות `resultOffset`. מבוקשים רק מעט השדות הדרושים לניתוח, וכל עמוד עובר אימות מול מעטפת שגיאה של ArcGIS לפני הפענוח.

In [ ]:
SOCIO_FIELDS = ['SEMEL_YISH', 'STAT11', 'YISHUV_STA', 'Shem_Yishuv', 'Shem_Yishuv_English',
                'Pop_Total', 'eshkol_mad', 'CLUSTER_2021', 'INDEX_VALUE_2021']

def cbs_query_url(offset=None, count_only=False):
    params = {'where': '1=1', 'f': 'json' if count_only else 'geojson'}
    if count_only:
        params['returnCountOnly'] = 'true'
    else:
        params.update({
            'outFields': ','.join(SOCIO_FIELDS),
            'returnGeometry': 'true',
            'outSR': '4326',
            'resultRecordCount': str(CBS_PAGE_SIZE),
            'resultOffset': str(offset or 0),
        })
    return CBS_LAYER_URL + '/query?' + urlencode(params, safe=',')

def _get_json(url):
    response = requests.get(url, timeout=180)
    response.raise_for_status()
    payload = response.json()
    if isinstance(payload, dict) and 'error' in payload:
        raise RuntimeError('CBS service returned an error: ' + json.dumps(payload['error'], ensure_ascii=False))
    return payload

def download_cbs_layer():
    if CBS_CACHE.exists():
        print('Using the cached CBS download (no network needed):', CBS_CACHE)
        return gpd.read_file(CBS_CACHE)
    total = int(_get_json(cbs_query_url(count_only=True))['count'])
    print('CBS layer reports %d statistical areas; downloading in pages of %d...' % (total, CBS_PAGE_SIZE))
    pages = []
    for offset in range(0, total, CBS_PAGE_SIZE):
        print('  features %d-%d / %d' % (offset + 1, min(offset + CBS_PAGE_SIZE, total), total))
        payload = _get_json(cbs_query_url(offset=offset))
        pages.append(gpd.GeoDataFrame.from_features(payload['features'], crs='EPSG:4326'))
    raw = gpd.GeoDataFrame(pd.concat(pages, ignore_index=True), geometry='geometry', crs='EPSG:4326')
    try:
        raw.to_file(CBS_CACHE, driver='GeoJSON')
        print('Cached the raw layer to', CBS_CACHE)
    except Exception as exc:
        print('WARNING: could not cache the layer (' + str(exc) + '); the next run will download again.')
    return raw

raw_cbs = download_cbs_layer()
print('downloaded polygons:', len(raw_cbs))
print('fields             :', [c for c in raw_cbs.columns if c != 'geometry'])

## נרמול השכבה הסוציו-אקונומית

השירות חושף את אותה תכונה תחת מספר שמות (כינויים בעברית ובאנגלית, גרסאות 2019 ו-2021), ולכן כל שדה ניתוח נלקח מעמודת המקור הראשונה הנושאת נתונים בפועל. לאחר מכן השורות מוגבלות לפוליגונים בעלי אשכול סוציו-אקונומי תקף בטווח 1-10 וגיאומטריה ממשית; פוליגונים ללא אשכול (אזורים שאינם למגורים, יישובים חדשים אחדים) אינם יכולים לתרום להשוואת שוויון.

כל מזהה נשמר כ-float או כמחרוזת פשוטה ולא כטיפוס `Int64` בר-ערכי-חסר של pandas, משום שעמודות שלמים ממוסכות זורקות שגיאה ב-`pd.cut`, במיסוך בוליאני ובתוך `scipy` - מקור אמיתי לתקלות בסקריפט המקורי.

In [ ]:
def first_existing(frame, columns):
    """First column that exists, filled in from the later ones where it is null."""
    values = None
    for col in columns:
        if col in frame.columns:
            values = frame[col].copy() if values is None else values.where(values.notna(), frame[col])
    if values is None:
        return pd.Series(np.nan, index=frame.index, dtype='object')
    return values

socio = gpd.GeoDataFrame({
    'socio_locality_code': first_existing(raw_cbs, ['SEMEL_YISH', 'סמל_יישוב']),
    'socio_stat_area_code': first_existing(raw_cbs, ['STAT11', 'סמל_אזור_סטטיסטי']),
    'socio_unit_full_code': first_existing(raw_cbs, ['YISHUV_STA']),
    'socio_locality': first_existing(raw_cbs, ['Shem_Yishuv', 'שם_יישוב']),
    'socio_locality_en': first_existing(raw_cbs, ['Shem_Yishuv_English']),
    'socio_population': first_existing(raw_cbs, ['Pop_Total', 'אוכלוסיית_המדד']),
    'socio_cluster': first_existing(raw_cbs, ['eshkol_mad', 'CLUSTER_2021', 'אשכול']),
    'socio_index_value': first_existing(raw_cbs, ['INDEX_VALUE_2021', 'ערך_מדד']),
}, geometry=raw_cbs.geometry, crs=raw_cbs.crs)

for col in ['socio_locality_code', 'socio_stat_area_code', 'socio_unit_full_code',
            'socio_population', 'socio_cluster', 'socio_index_value']:
    socio[col] = pd.to_numeric(socio[col], errors='coerce')

def code_to_str(series):
    return series.map(lambda v: '' if pd.isna(v) else str(int(v)))

socio['socio_unit_id'] = code_to_str(socio['socio_unit_full_code'])
fallback_id = code_to_str(socio['socio_locality_code']) + '_' + code_to_str(socio['socio_stat_area_code'])
no_full_code = socio['socio_unit_full_code'].isna()
socio.loc[no_full_code, 'socio_unit_id'] = fallback_id[no_full_code]

keep = socio['socio_cluster'].between(1, 10).fillna(False) & socio.geometry.notna()
socio = socio.loc[keep].copy()

HAS_INDEX_VALUE = bool(socio['socio_index_value'].notna().any())
print('usable statistical areas   :', len(socio))
print('with a population figure   : %d (%.1f%%)'
      % (int(socio['socio_population'].notna().sum()), 100 * socio['socio_population'].notna().mean()))
print('continuous index available :', HAS_INDEX_VALUE)
if not HAS_INDEX_VALUE:
    print('  -> the service now serves INDEX_VALUE_2021 as NULL; only the 1-10 cluster is usable.')
print()
print(socio['socio_cluster'].value_counts().sort_index().to_string())

## שיוך תחנות לאזורים סטטיסטיים

שני מעברים, בסדר הזה:

1. **נקודה בתוך פוליגון** (`predicate='within'`) - השיוך המדויק; תחנה הנמצאת בתוך אזור סטטיסטי שייכת אליו.
2. **הפוליגון הקרוב ביותר**, מוגבל ל-`NEAREST_JOIN_MAX_DISTANCE_M`, עבור התחנות שאינן נופלות בשום פוליגון. שכבת הלמ"ס מכסה אזורי מגורים בלבד, ולכן תחנות בכבישים בין-עירוניים, באזורי תעשייה ובמחלפים נופלות באופן לגיטימי מחוץ לכל פוליגון; שיוכן לשכונה הקרובה ביותר הוא קירוב מתועד, והמרחק נשמר לכל תחנה (`socio_join_distance_m`) כך שניתן לסנן את הקירוב בהמשך.

המרחקים נמדדים לאחר היטל ל-EPSG:3857, שהוא מטרי בקירוב בקו הרוחב של ישראל (ניפוח קנה מידה של פי ~1.18) - ולכן המגבלה שמרנית ולא מדויקת. מילון איכות השיוך מתעד את שיעור ההתאמה ואת התפלגות המרחקים.

In [ ]:
stops_gdf = gpd.GeoDataFrame(stops.copy(),
                             geometry=gpd.points_from_xy(stops['lon'], stops['lat']),
                             crs='EPSG:4326')

socio_cols = ['socio_unit_id', 'socio_locality_code', 'socio_stat_area_code', 'socio_locality',
              'socio_locality_en', 'socio_population', 'socio_cluster', 'socio_index_value', 'geometry']

joined = gpd.sjoin(stops_gdf, socio[socio_cols], how='left', predicate='within')
joined = joined[~joined.index.duplicated(keep='first')].drop(columns=['index_right'], errors='ignore')
joined['socio_join_method'] = np.where(joined['socio_cluster'].notna(), 'within', None)
joined['socio_join_distance_m'] = np.where(joined['socio_cluster'].notna(), 0.0, np.nan)

missing_index = joined.index[joined['socio_cluster'].isna()]
print('stops inside a polygon      :', int((joined['socio_join_method'] == 'within').sum()))
print('stops needing nearest join  :', len(missing_index))

if len(missing_index):
    nearest = gpd.sjoin_nearest(
        stops_gdf.loc[missing_index].to_crs(3857),
        socio[socio_cols].to_crs(3857),
        how='left',
        max_distance=NEAREST_JOIN_MAX_DISTANCE_M,
        distance_col='socio_join_distance_m',
    )
    nearest = nearest[~nearest.index.duplicated(keep='first')].drop(columns=['index_right'], errors='ignore')
    matched_nearest = nearest.index[nearest['socio_cluster'].notna()]
    for col in [c for c in socio_cols if c != 'geometry'] + ['socio_join_distance_m']:
        joined.loc[matched_nearest, col] = nearest.loc[matched_nearest, col]
    joined.loc[matched_nearest, 'socio_join_method'] = 'nearest'

joined['socio_cluster'] = pd.to_numeric(joined['socio_cluster'], errors='coerce')
joined['socio_cluster_group'] = pd.cut(joined['socio_cluster'], bins=[0, 4, 7, 10],
                                       labels=['low_1_4', 'middle_5_7', 'high_8_10'])

nearest_dist = joined.loc[joined['socio_join_method'] == 'nearest', 'socio_join_distance_m']
join_quality = {
    'stop_metrics_source': str(STOP_METRICS_PATH),
    'cbs_layer': CBS_LAYER_URL,
    'cbs_layer_is_external_runtime_dependency': True,
    'total_stops': int(len(joined)),
    'matched_within_polygon': int((joined['socio_join_method'] == 'within').sum()),
    'matched_by_nearest_polygon': int((joined['socio_join_method'] == 'nearest').sum()),
    'unmatched': int(joined['socio_join_method'].isna().sum()),
    'nearest_join_max_distance_m': NEAREST_JOIN_MAX_DISTANCE_M,
    'nearest_join_median_distance_m': None if nearest_dist.empty else round(float(nearest_dist.median()), 1),
    'nearest_join_max_observed_distance_m': None if nearest_dist.empty else round(float(nearest_dist.max()), 1),
}
join_quality['match_rate_pct'] = round(
    100 * (join_quality['matched_within_polygon'] + join_quality['matched_by_nearest_polygon'])
    / max(join_quality['total_stops'], 1), 2)

print()
print(json.dumps(join_quality, ensure_ascii=False, indent=2))

## צבירה לרמת שכונות (אזורים סטטיסטיים של הלמ"ס)

האזור הסטטיסטי הוא יחידת הניתוח הנכונה, משתי סיבות:

1. האשכול הסוציו-אקונומי הוא **תכונה של האזור**, לא של התחנה. תיאום שלו ברמת התחנה יוצר פסאודו-שכפול (pseudo-replication): אזור מרכז עיר צפוף עם 90 תחנות יימנה 90 פעמים מול פרוור עם 4 תחנות, אך ורק משום שיש בו יותר תחנות.
2. שירות לנפש קיים רק ברמת האזור, משום שנתון האוכלוסייה משויך לאזור.

עבור כל אזור אנו סופרים תחנות, מסכמים את עצירות הנסיעות המתוכננות, לוקחים את ממוצע דרגת גרף הנסיעות ואת חלקן של התחנות הקריטיות, ומתעדים את קהילת Louvain / היישוב / האזור / המטרופולין הדומיננטיים לפי שכיח. `stop_use_per_1000` - עצירות מתוכננות לכל 1,000 תושבים - הוא משתנה השירות שישמש מכאן ואילך.

In [ ]:
matched = joined[joined['socio_cluster'].notna()].copy()
matched['active_stop'] = matched['stop_use_count'] > 0
matched['low_socio_stop'] = matched['socio_cluster'] <= 4

nb = (matched.groupby('socio_unit_id')
      .agg(socio_cluster=('socio_cluster', 'first'),
           population=('socio_population', 'first'),
           stops=('stop_id', 'count'),
           active_stops=('active_stop', 'sum'),
           total_stop_use=('stop_use_count', 'sum'),
           avg_trip_graph_degree=('trip_graph_degree', 'mean'),
           critical_stops=('is_critical', 'sum'),
           community=('community', mode_or_na),
           city=('socio_locality', mode_or_na),
           region=('region', mode_or_na),
           metro=('metro', mode_or_na))
      .reset_index())

nb['socio_cluster'] = pd.to_numeric(nb['socio_cluster'], errors='coerce')
nb['community'] = pd.to_numeric(nb['community'], errors='coerce')
nb['stop_use_per_1000'] = per_1000(nb['total_stop_use'], nb['population'])
nb['stops_per_1000'] = per_1000(nb['stops'], nb['population'])
nb['stop_use_per_stop'] = np.where(nb['stops'] > 0, nb['total_stop_use'] / nb['stops'], np.nan)
nb['critical_stop_share'] = nb['critical_stops'] / nb['stops']
nb['active_stop_share'] = nb['active_stops'] / nb['stops']

print('neighbourhoods (statistical areas) reached by the network :', len(nb))
print('  with a usable per-capita figure                         :', int(nb['stop_use_per_1000'].notna().sum()))
print('  with a Louvain community                                :', int(nb['community'].notna().sum()))
nb.head()

## התמונה הארצית

תחילה השאלה הנאיבית: במצרף על פני המדינה כולה, האם השירות לנפש תלוי באשכול הסוציו-אקונומי? מופקים שני מבטים.

- **סיכום לפי אשכול למ"ס** (1-10): אוכלוסייה, תחנות, עצירות, והשירות לנפש של השכונה החציונית. החציונים מדווחים לצד יחסי המצרף משום שקומץ שכונות של תחנות מרכזיות עם אוכלוסיית מגורים זעירה שולט בכל ממוצע.
- **טבלת מתאמי Spearman** בשני היקפים: רמת השכונה (הבר-הגנה) ורמת התחנה (נשמרת לצורך השוואה לעבודה הקודמת, ומסומנת כפסאודו-משוכפלת).

נעשה שימוש ב-Spearman ולא ב-Pearson משום שהאשכול הסוציו-אקונומי הוא סולם סודר בטווח 1-10 והשירות לנפש מוטה ימינה בצורה קיצונית.

In [ ]:
cluster_summary = (nb.dropna(subset=['socio_cluster'])
                   .groupby('socio_cluster')
                   .agg(neighborhoods=('socio_unit_id', 'count'),
                        population=('population', 'sum'),
                        stops=('stops', 'sum'),
                        active_stops=('active_stops', 'sum'),
                        total_stop_use=('total_stop_use', 'sum'),
                        median_stops_per_1000=('stops_per_1000', 'median'),
                        median_stop_use_per_1000=('stop_use_per_1000', 'median'),
                        mean_trip_graph_degree=('avg_trip_graph_degree', 'mean'),
                        mean_critical_stop_share=('critical_stop_share', 'mean'))
                   .reset_index())
cluster_summary['stops_per_1000_residents'] = per_1000(cluster_summary['stops'], cluster_summary['population'])
cluster_summary['stop_use_per_1000_residents'] = per_1000(cluster_summary['total_stop_use'], cluster_summary['population'])
cluster_summary['stop_use_per_stop'] = np.where(cluster_summary['stops'] > 0,
                                                cluster_summary['total_stop_use'] / cluster_summary['stops'], np.nan)
cluster_summary['active_stop_share'] = cluster_summary['active_stops'] / cluster_summary['stops']

national_rows = []
for metric in ['stop_use_per_1000', 'stops_per_1000', 'stop_use_per_stop',
               'avg_trip_graph_degree', 'critical_stop_share']:
    rho, p_value, n = spearman(nb['socio_cluster'], nb[metric])
    national_rows.append({'scope': 'neighborhood (statistical area)', 'metric': metric,
                          'spearman_rho': rho, 'p_value': p_value, 'n': n,
                          'direction': describe_rho(rho)})
for metric in ['stop_use_count', 'trip_graph_degree', 'is_critical']:
    rho, p_value, n = spearman(matched['socio_cluster'], matched[metric].astype(float))
    national_rows.append({'scope': 'stop (pseudo-replicated)', 'metric': metric,
                          'spearman_rho': rho, 'p_value': p_value, 'n': n,
                          'direction': describe_rho(rho)})
national_correlations = pd.DataFrame(national_rows)

NATIONAL_RHO, NATIONAL_P, NATIONAL_N = spearman(nb['socio_cluster'], nb['stop_use_per_1000'])
print('NATIONAL pooled rho(socioeconomic cluster, stop calls per 1,000) = %.3f  (p = %.3g, n = %d)'
      % (NATIONAL_RHO, NATIONAL_P, NATIONAL_N))
print('direction:', describe_rho(NATIONAL_RHO), '- but note how small the effect is.')
print()
print(national_correlations.to_string(index=False))
print()
print(cluster_summary[['socio_cluster', 'neighborhoods', 'population', 'stops',
                       'median_stops_per_1000', 'median_stop_use_per_1000']].to_string(index=False))

### איור תומך: שירות לפי אשכול סוציו-אקונומי

תרשים תיאורי פשוט של הטבלה שלעיל. מוצגות עמודות של חציון-השכונות ולא של יחס המצרף, משום שהמצרף נשלט על ידי מעט שכונות המארחות תחנות מרכזיות בעוד שכמעט איש אינו מתגורר בהן. העמודות קרובות למישוריות, וזו בדיוק תמצית הסיפור הארצי: ברזולוציה זו הרשת נראית שוויונית באופן כללי.

In [ ]:
plot_data = cluster_summary.dropna(subset=['socio_cluster']).copy()
plot_data['socio_cluster'] = plot_data['socio_cluster'].astype(int)
palette = sns.color_palette('crest', n_colors=len(plot_data))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(plot_data['socio_cluster'], plot_data['median_stops_per_1000'], color=palette)
axes[0].set_title('Median neighbourhood: stops per 1,000 residents')
axes[0].set_xlabel('CBS socioeconomic cluster (1 = weakest, 10 = strongest)')
axes[0].set_ylabel('Stops / 1,000 residents')
axes[1].bar(plot_data['socio_cluster'], plot_data['median_stop_use_per_1000'], color=palette)
axes[1].set_title('Median neighbourhood: scheduled stop calls per 1,000 residents')
axes[1].set_xlabel('CBS socioeconomic cluster (1 = weakest, 10 = strongest)')
axes[1].set_ylabel('Stop calls / 1,000 residents')
for ax in axes:
    ax.set_xticks(plot_data['socio_cluster'])
fig.suptitle('National view: service level by socioeconomic cluster (medians over neighbourhoods)')
fig.tight_layout()
save_fig(fig, 'socioeconomic_access_by_cluster.png')

## קהילות Louvain כפרוקסי מבוסס-נתונים לערים ולמטרופולינים

כדי לבחון את הפרדוקס דרושות קבוצות שבתוכן ההשוואה "שכונה חזקה מול שכונה חלשה" הוגנת - כלומר קבוצות המשורתות על ידי תת-רשת תפעולית אחת. גבולות עירוניים מנהליים אינם נמצאים בקובץ ה-GTFS, אך קהילות Louvain של גרף הנסיעות כן, והן מתיישבות היטב עם ערים ועם מסדרונות מטרופוליניים משום שרוב הנסיעות נותרות בתוך אחד מהם.

לפיכך כל קהילה מסוכמת ו**מתויגת מתוך הנתונים**: התווית היא היישוב השכיח של הלמ"ס בקרב תחנות הקהילה. הסקריפט הקודם קיבע בקוד מזהי קהילות (קהילה 34 = נתניה, 23 = בית שאן), מה שמתייג באופן שגוי ושקט כל איור ברגע שזיהוי הקהילות מורץ מחדש עם seed שונה או עם גרף שונה. שום דבר כאן אינו תלוי במזהה קהילה מספרי.

In [ ]:
with_community = matched[matched['community'].notna()].copy()

community_summary = (with_community.groupby('community')
                     .agg(stops=('stop_id', 'count'),
                          active_stops=('active_stop', 'sum'),
                          total_stop_use_count=('stop_use_count', 'sum'),
                          avg_stop_use_count=('stop_use_count', 'mean'),
                          avg_trip_graph_degree=('trip_graph_degree', 'mean'),
                          avg_socio_cluster=('socio_cluster', 'mean'),
                          median_socio_cluster=('socio_cluster', 'median'),
                          dominant_socio_cluster=('socio_cluster', mode_or_na),
                          low_socio_stop_share=('low_socio_stop', 'mean'),
                          dominant_city=('socio_locality', mode_or_na),
                          dominant_region=('region', mode_or_na),
                          dominant_metro=('metro', mode_or_na),
                          socio_areas=('socio_unit_id', 'nunique'))
                     .reset_index())

covered_population = (nb.dropna(subset=['community'])
                      .groupby('community')['population'].sum()
                      .rename('covered_population_estimate').reset_index())
community_summary = community_summary.merge(covered_population, on='community', how='left')
community_summary['active_stop_share'] = community_summary['active_stops'] / community_summary['stops']
community_summary['use_per_1000'] = per_1000(community_summary['total_stop_use_count'],
                                             community_summary['covered_population_estimate'])
community_summary['community_label'] = [
    ('%s (#%d)' % (city, int(cid))) if isinstance(city, str) else ('community #%d' % int(cid))
    for city, cid in zip(community_summary['dominant_city'], community_summary['community'])]
community_summary = community_summary.sort_values('stops', ascending=False).reset_index(drop=True)

print('communities with socioeconomic coverage:', len(community_summary))
print()
print(community_summary[['community_label', 'stops', 'socio_areas', 'avg_socio_cluster',
                         'use_per_1000', 'dominant_region']].head(12).to_string(index=False))

## שלב א' של הסיפור - נקודה אחת לכל קהילה: אין כלל נראה לעין

איור המצגת הראשון. כל נקודה היא קהילה שלמה, הממוקמת לפי הרמה הסוציו-אקונומית **הממוצעת** שלה ולפי השירות לנפש **הממוצע** שלה. לענן אין שיפוע שמיש: מיצוע קהילה מערבב יחד את שכונותיה החזקות והחלשות, וההבדלים הבין-קהילתיים שנותרים מונעים על ידי צפיפות וגיאוגרפיה, לא על ידי עושר.

הקו המקווקו הוא התאמת OLS על **כל** הנקודות המשורטטות, כלומר בדיוק השורות שמאחורי ערך ה-Spearman rho המצוטט. ציר ה-y נחתך באחוזון ה-95 כדי שהענן יהיה קריא; החיתוך משפיע על התצוגה בלבד, לעולם לא על ההתאמה או על הסטטיסטיקה.

פלט: `figures/socioeconomic_cluster_average_flat.png`.

In [ ]:
flat = community_summary.dropna(subset=['avg_socio_cluster', 'use_per_1000']).copy()
rho_flat, p_flat, n_flat = spearman(flat['avg_socio_cluster'], flat['use_per_1000'])
y_view = float(np.nanpercentile(flat['use_per_1000'], 95))

fig, ax = plt.subplots(figsize=(12, 7))
ax.scatter(flat['avg_socio_cluster'], flat['use_per_1000'],
           s=np.clip(flat['stops'], 30, 600), alpha=0.55,
           color='#34557a', edgecolor='white', linewidth=0.6)
coef_flat = ols_slope(flat['avg_socio_cluster'], flat['use_per_1000'])
if coef_flat is not None:
    xs = np.array([flat['avg_socio_cluster'].min(), flat['avg_socio_cluster'].max()])
    ax.plot(xs, np.polyval(coef_flat, xs), color='#c0392b', linewidth=3, linestyle='--',
            label='OLS fit on all %d communities (untrimmed)' % len(flat))
ax.set_ylim(0, 1.05 * y_view)
ax.set_xlim(0.5, 10.5)
ax.set_title('Each point is one Louvain community (%d communities)' % len(flat),
             fontsize=16, fontweight='bold', pad=12)
ax.set_xlabel('Mean socioeconomic cluster of the community  (1 = weakest, 10 = strongest)')
ax.set_ylabel('Service per capita (scheduled stop calls / 1,000 residents)')
ax.legend(loc='upper left')
ax.annotate('Spearman rho = %.2f,  p = %.2f  (n = %d communities)' % (rho_flat, p_flat, n_flat)
            + '\nBubble size = stops in the community'
            + '\nY axis clipped at the 95th percentile for readability;'
            + '\nthe fit and the rho use every point.',
            xy=(0.98, 0.95), xycoords='axes fraction', ha='right', va='top', fontsize=12,
            bbox=dict(boxstyle='round,pad=0.5', fc='#f4f4f4', ec='#bbbbbb', lw=1.0))
save_fig(fig, 'socioeconomic_cluster_average_flat.png')
print('community-level rho = %.3f (p = %.3f) -> %s'
      % (rho_flat, p_flat, 'significant' if p_flat < ALPHA else 'NOT significant'))

## אותה נקודה, מסופרת באמצעות ערים מזוהות בשמן

תרשים הפיזור שלעיל משכנע סטטיסטיקאי ובלתי נראה לכל השאר, ולכן אותה עובדה מנוסחת מחדש באמצעות מקומות מוכרים: קהילות מסודרות מהענייה ביותר לעשירה ביותר, עם השירות לנפש שלהן על העמודות. גובה העמודות קופץ ללא כל קשר לסדר.

**בחירה.** הסקריפט המקורי קיבע בקוד שישה שמות ערים. בחירות מקובעות בקוד הן ליקוט דובדבנים (cherry-picking) אלא אם הן מנומקות, ולכן כאן הערים נלקחות מתוך הנתונים: `N_CITIES_NO_RULE` הקהילות בעלות מספר התחנות הגדול ביותר, לאחר הסרת כפילויות לפי העיר הדומיננטית, וממוינות לפי הרמה הסוציו-אקונומית. יש להגדיר את `CITIES_IN_NO_RULE_FIGURE` כרשימת שמות יישובים כדי לשחזר שקופית אוצרותית מסוימת במקום זאת.

פלט: `figures/socioeconomic_cities_no_rule.png`.

In [ ]:
available = community_summary.dropna(subset=['avg_socio_cluster', 'use_per_1000', 'dominant_city'])
if CITIES_IN_NO_RULE_FIGURE:
    picked = (available[available['dominant_city'].isin(CITIES_IN_NO_RULE_FIGURE)]
              .drop_duplicates('dominant_city'))
    selection_note = 'Cities chosen by hand (CITIES_IN_NO_RULE_FIGURE)'
else:
    picked = (available.sort_values('stops', ascending=False)
              .drop_duplicates('dominant_city')
              .head(N_CITIES_NO_RULE))
    selection_note = 'The %d largest communities by stop count (no hand-picking)' % N_CITIES_NO_RULE
picked = picked.sort_values('avg_socio_cluster')

fig, ax = plt.subplots(figsize=(12, 7))
norm = plt.Normalize(1, 10)
colors = plt.cm.RdYlGn(norm(picked['avg_socio_cluster']))
bars = ax.bar(range(len(picked)), picked['use_per_1000'], color=colors,
              edgecolor='#333333', linewidth=0.8)
ax.set_xticks(range(len(picked)))
ax.set_xticklabels(['%s\n(cluster %.1f)' % (c, s)
                    for c, s in zip(picked['dominant_city'], picked['avg_socio_cluster'])],
                   fontsize=12)
for bar, val in zip(bars, picked['use_per_1000']):
    ax.text(bar.get_x() + bar.get_width() / 2, val, format(int(round(val)), ','),
            ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.set_ylabel('Service per capita (scheduled stop calls / 1,000 residents)')
ax.set_title('Ordered from the poorest community to the richest - service jumps with no rule',
             fontsize=15, fontweight='bold', pad=14)
ax.annotate('Socioeconomic level does not predict service\nSpearman rho = %.2f,  p = %.2f  (%s, all %d communities)'
            % (rho_flat, p_flat, 'significant' if p_flat < ALPHA else 'not significant', n_flat)
            + '\n' + selection_note,
            xy=(0.98, 0.95), xycoords='axes fraction', ha='right', va='top',
            fontsize=13, fontweight='bold', color='#c0392b',
            bbox=dict(boxstyle='round,pad=0.5', fc='#fdecea', ec='#c0392b', lw=1.5))
ax.margins(y=0.20)
save_fig(fig, 'socioeconomic_cities_no_rule.png')
print(picked[['community_label', 'stops', 'avg_socio_cluster', 'use_per_1000']].to_string(index=False))

## שלב ב' - המבחן בתוך הקהילה, עם בקרה על השוואות מרובות

כעת המבחן האמיתי. עבור כל קהילה שיש בה לפחות `MIN_NEIGHBORHOODS` שכונות הפרוסות על פני לפחות `MIN_DISTINCT_SOCIO_LEVELS` רמות סוציו-אקונומיות, אנו מתאמים את האשכול הסוציו-אקונומי של השכונה עם השירות לנפש שלה. זהו מתאם *בתוך קבוצה*: הוא משווה שכונה חזקה לשכונה חלשה **המשורתות על ידי אותה תת-רשת**, וזו ההשוואה שהמספר הארצי המצרפי אינו יכול לבצע.

**באג שתוקן כאן: השוואות מרובות.** כ-50-60 קהילות נבחנות בו-זמנית. ברמת מובהקות alpha = 0.05 די בכך כדי להפיק בתוחלת כשלוש תוצאות "מובהקות" מרעש בלבד, והגרסה הקודמת דיווחה את ערכי ה-p הגולמיים ללא כל תיקון. מחברת זו מדווחת, זה לצד זה:

- `p_use_per_capita` - ערך ה-p הגולמי;
- `p_fdr_bh` - מתוקן לפי Benjamini-Hochberg (שולט בשיעור התגליות השקריות, התיקון המתאים לסינון מסוג זה);
- `p_bonferroni` - מתוקן לפי Bonferroni (שולט בשיעור השגיאה המשפחתי; שמרני).

האיורים מסמנים מובהקות באמצעות הערך **המתוקן לפי FDR**, וערך ה-p הגולמי נותר בטבלה כך ששום דבר אינו מוסתר. Benjamini-Hochberg ממומש בקוד עצמו (step-up על ערכי ה-p הממוינים עם מעבר מונוטוניות) כדי להימנע מתלות נוספת.

In [ ]:
def benjamini_hochberg(p_values):
    """Benjamini-Hochberg FDR-adjusted p-values; NaNs pass through as NaN."""
    p = np.asarray(p_values, dtype=float)
    out = np.full(p.shape, np.nan)
    ok = ~np.isnan(p)
    pv = p[ok]
    m = pv.size
    if m == 0:
        return out
    order = np.argsort(pv)
    ranked = pv[order]
    adjusted = ranked * m / np.arange(1, m + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]   # enforce monotonicity
    adjusted = np.clip(adjusted, 0, 1)
    restored = np.empty(m)
    restored[order] = adjusted
    out[ok] = restored
    return out

rows = []
for community, group in nb.dropna(subset=['community']).groupby('community'):
    group = group.dropna(subset=['socio_cluster'])
    if len(group) < MIN_NEIGHBORHOODS or group['socio_cluster'].nunique() < MIN_DISTINCT_SOCIO_LEVELS:
        continue
    rho_use, p_use, n_use = spearman(group['socio_cluster'], group['stop_use_per_1000'])
    rho_stops, p_stops, _ = spearman(group['socio_cluster'], group['stops_per_1000'])
    stop_group = with_community[with_community['community'] == community]
    rho_stop_level, p_stop_level, _ = spearman(stop_group['socio_cluster'], stop_group['stop_use_count'])
    city = mode_or_na(group['city'])
    rows.append({
        'community': int(community),
        'dominant_city': city,
        'community_label': ('%s (#%d)' % (city, int(community))) if isinstance(city, str) else ('community #%d' % int(community)),
        'dominant_region': mode_or_na(group['region']),
        'dominant_metro': mode_or_na(group['metro']),
        'n_neighborhoods': int(len(group)),
        'n_neighborhoods_with_population': int(n_use),
        'n_stops': int(len(stop_group)),
        'population': float(group['population'].sum()),
        'mean_socio_cluster': float(group['socio_cluster'].mean()),
        'socio_cluster_spread': int(group['socio_cluster'].nunique()),
        'rho_use_per_capita': rho_use,
        'p_use_per_capita': p_use,
        'rho_stops_per_capita': rho_stops,
        'p_stops_per_capita': p_stops,
        'rho_stop_level_use': rho_stop_level,
        'p_stop_level_use': p_stop_level,
    })

within = pd.DataFrame(rows)
if within.empty:
    raise RuntimeError('No community passed the within-community thresholds - check the community coverage above.')

N_TESTS = int(within['p_use_per_capita'].notna().sum())
within['abs_rho'] = within['rho_use_per_capita'].abs()
within['p_fdr_bh'] = benjamini_hochberg(within['p_use_per_capita'])
within['p_bonferroni'] = np.clip(within['p_use_per_capita'] * N_TESTS, 0, 1)
within['significant_raw_05'] = within['p_use_per_capita'] < ALPHA
within['significant_fdr_05'] = within['p_fdr_bh'] < ALPHA
within['significant_bonferroni_05'] = within['p_bonferroni'] < ALPHA
within['direction'] = [describe_rho(r) for r in within['rho_use_per_capita']]
within = within.sort_values('rho_use_per_capita').reset_index(drop=True)

valid = within.dropna(subset=['rho_use_per_capita'])
print('communities tested                       : %d' % N_TESTS)
print('rho range                                : %.2f .. %.2f'
      % (valid['rho_use_per_capita'].min(), valid['rho_use_per_capita'].max()))
print('significant at raw p < %.2f               : %d' % (ALPHA, int(within['significant_raw_05'].sum())))
print('significant after Benjamini-Hochberg FDR : %d' % int(within['significant_fdr_05'].sum()))
print('significant after Bonferroni             : %d' % int(within['significant_bonferroni_05'].sum()))
print('expected false positives at raw alpha    : %.1f' % (ALPHA * N_TESTS))
print()
print('Strongest negative rho (service tilted to the weaker neighbourhoods):')
print(valid.head(5)[['community_label', 'dominant_region', 'n_neighborhoods',
                     'rho_use_per_capita', 'p_use_per_capita', 'p_fdr_bh']].to_string(index=False))
print()
print('Strongest positive rho (service tilted to the stronger neighbourhoods):')
print(valid.tail(5)[['community_label', 'dominant_region', 'n_neighborhoods',
                     'rho_use_per_capita', 'p_use_per_capita', 'p_fdr_bh']].to_string(index=False))

## איור: מתאמים בתוך הקהילות מול הקו הארצי

עמודה אחת לכל קהילה, המציגה את **ערך ה-Spearman rho הגולמי** (ראו את פרק מוסכמת הסימן: שלילי = השירות נוטה לטובת השכונות החלשות, ירוק; חיובי = נוטה לטובת החזקות, אדום). הקו הכחול המקווקו הוא ה-rho הארצי המצרפי, הנמצא קרוב לאפס בעוד שקהילות בודדות מגיעות רחוק לשני הכיוונים - פער זה הוא הפרדוקס.

שתי הערות יושרה משורטטות על האיור עצמו:
- העמודות **דהויות כאשר הקהילה אינה מובהקת לאחר תיקון FDR**, כך שקורא לא יטעה לחשוב שקהילה קטנה ורועשת מהווה ממצא.
- הפאנל מציג את `TOP_COMMUNITIES_FOR_FIGURE` הקהילות בעלות **ערך ה-rho המוחלט הגדול ביותר** מבין אלה שיש בהן לפחות `FIG_MIN_NEIGHBORHOODS` שכונות. זו בחירה על פי גודל האפקט ולכן היא מגזימה את הפיזור האופייני; הרשימה המלאה והבלתי-נבחרת נמצאת ב-`tables/socioeconomic_within_cluster_correlation.csv`.

פלט: `figures/socioeconomic_within_cluster_correlation.png`.

In [ ]:
bar_data = valid[valid['n_neighborhoods'] >= FIG_MIN_NEIGHBORHOODS].copy()
bar_data = bar_data.reindex(bar_data['abs_rho'].sort_values(ascending=False).index)
bar_data = bar_data.head(TOP_COMMUNITIES_FOR_FIGURE).sort_values('rho_use_per_capita')

labels = ['%s  (%d neighbourhoods)' % (c, n)
          for c, n in zip(bar_data['dominant_city'], bar_data['n_neighborhoods'])]
colors = [rho_color(v) for v in bar_data['rho_use_per_capita']]

fig, ax = plt.subplots(figsize=(13, 8))
y = np.arange(len(bar_data))
bars = ax.barh(y, bar_data['rho_use_per_capita'], color=colors, edgecolor='white')
for bar, sig in zip(bars, bar_data['significant_fdr_05']):
    if not sig:
        bar.set_alpha(0.35)
ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=11)
ax.axvline(0, color='#555555', linewidth=1)
ax.axvline(NATIONAL_RHO, color='#2471a3', linestyle='--', linewidth=2.5,
           label='national pooled rho = %.2f' % NATIONAL_RHO)
for yi, v in enumerate(bar_data['rho_use_per_capita']):
    ax.text(v + (0.015 if v >= 0 else -0.015), yi, '%+.2f' % v, va='center',
            ha='left' if v >= 0 else 'right', fontsize=11, fontweight='bold')
lo = float(bar_data['rho_use_per_capita'].min())
hi = float(bar_data['rho_use_per_capita'].max())
ax.set_xlim(lo - 0.20, hi + 0.20)
ax.set_xlabel(RHO_AXIS_LABEL + '\nraw rho, no sign flipping   |   ' + RHO_NEG_MEANING
              + '   |   ' + RHO_POS_MEANING, fontsize=11)
ax.set_title('The socioeconomic service gap changes sign from community to community\n'
             'green = weaker neighbourhoods get more service per capita, red = stronger ones do',
             fontsize=14, pad=12)
ax.legend(loc='lower right', fontsize=11)
ax.text(0.02, 0.97,
        'Faded bar = not significant after Benjamini-Hochberg FDR correction\n'
        '(%d of %d tested communities survive FDR; raw p < %.2f would pass %d)\n'
        'Panel shows the %d largest |rho| with >= %d neighbourhoods - a selection on effect size'
        % (int(within['significant_fdr_05'].sum()), N_TESTS, ALPHA,
           int(within['significant_raw_05'].sum()), len(bar_data), FIG_MIN_NEIGHBORHOODS),
        transform=ax.transAxes, ha='left', va='top', fontsize=11, color='#333333',
        bbox=dict(boxstyle='round,pad=0.4', fc='#f4f4f4', ec='#cccccc', lw=1.0))
save_fig(fig, 'socioeconomic_within_cluster_correlation.png')

## איור: שלוש קהילות לדוגמה (קיצוני בחירה, לא מקרים אופייניים)

שלושה פאנלים של תרשימי פיזור, שכונה אחת לכל נקודה. הפאנלים נבחרים כ-**rho השלילי ביותר**, **ה-rho הקרוב ביותר לאפס** ו-**ה-rho החיובי ביותר** מבין קהילות שיש בהן לפחות `EXAMPLE_MIN_NEIGHBORHOODS` שכונות.

**יש לקרוא זאת כקיצון של בחירה, לא כמדגם מייצג.** בחירת ה-argmin וה-argmax של סטטיסטי על פני כ-50 מבחנים מבטיחה פאנלים בעלי מראה חזק אפילו תחת רעש טהור; זו בדיוק הסיבה שערך ה-p המתוקן לפי FDR מודפס בכותרת כל פאנל. תפקידו של האיור הוא להראות כיצד *נראה* קשר חזק בתוך קהילה, לא להעריך עד כמה קשר כזה נפוץ.

כל קו אדום הוא התאמת OLS על **כל** שכונות הקהילה - אותן שורות בדיוק כמו ה-rho שבכותרת. ציר ה-y נחתך לצורכי קריאות בלבד.

פלט: `figures/socioeconomic_within_cluster_examples.png`.

In [ ]:
pool = valid[valid['n_neighborhoods'] >= EXAMPLE_MIN_NEIGHBORHOODS]
if len(pool) < 3:
    print('Only %d communities have >= %d neighbourhoods; falling back to >= %d.'
          % (len(pool), EXAMPLE_MIN_NEIGHBORHOODS, FIG_MIN_NEIGHBORHOODS))
    pool = valid[valid['n_neighborhoods'] >= FIG_MIN_NEIGHBORHOODS]
pool = pool.sort_values('rho_use_per_capita')

most_negative = pool.iloc[0]
most_positive = pool.iloc[-1]
near_zero = pool.reindex(pool['rho_use_per_capita'].abs().sort_values().index).iloc[0]
picks = [('most negative rho', most_negative), ('rho closest to zero', near_zero),
         ('most positive rho', most_positive)]

fig, axes = plt.subplots(1, 3, figsize=(17, 5.4))
for ax, (why, row) in zip(axes, picks):
    group = nb[nb['community'] == row['community']].dropna(subset=['socio_cluster', 'stop_use_per_1000'])
    ax.scatter(group['socio_cluster'], group['stop_use_per_1000'],
               s=np.clip(group['population'] / 200, 12, 240), alpha=0.65,
               color='#2c3e50', edgecolor='white', linewidth=0.4)
    coef = ols_slope(group['socio_cluster'], group['stop_use_per_1000'])
    if coef is not None:
        xs = np.array([group['socio_cluster'].min(), group['socio_cluster'].max()])
        ax.plot(xs, np.polyval(coef, xs), color='#c0392b', linewidth=2)
    y_cap = float(np.nanpercentile(group['stop_use_per_1000'], 97))
    ax.set_ylim(-0.05 * y_cap, 1.10 * y_cap)
    ax.set_title('%s\nrho = %.2f   (p_raw = %.3g, p_FDR = %.3g, n = %d)\n[selected as: %s]'
                 % (row['community_label'], row['rho_use_per_capita'], row['p_use_per_capita'],
                    row['p_fdr_bh'], int(row['n_neighborhoods']), why), fontsize=11)
    ax.set_xlabel('Socioeconomic cluster (1 = weakest, 10 = strongest)')
    ax.set_ylabel('Stop calls per 1,000 residents')
fig.suptitle('Inside a single community: strong vs weak neighbourhoods and their service level\n'
             'these three panels are SELECTION EXTREMA (argmin / nearest-zero / argmax of rho), not typical communities',
             fontsize=13)
fig.tight_layout(rect=(0, 0, 1, 0.92))
save_fig(fig, 'socioeconomic_within_cluster_examples.png')

## איור: פרדוקס סימפסון, בתרשים אחד

השורה התחתונה. שתי קהילות שמגמותיהן הפנימיות מצביעות לכיוונים **מנוגדים** משורטטות מעל המגמה הארצית המצרפית. הקו הארצי כמעט מישורי לא משום שהשירות מחולק באופן שווה, אלא משום שגרדיאנטים מקומיים הפוכים מקזזים זה את זה כאשר השכונות ממוצעות יחד. זו ההגדרה הקלאסית של פרדוקס סימפסון, וזו הסיבה שמתאם ארצי יחיד הוא התשובה השגויה לשאלת השוויון.

שתי הקהילות **נבחרות מתוך הנתונים** - ה-rho השלילי ביותר וה-rho החיובי ביותר בתוך קהילה מבין אלה שיש בהן לפחות `FIG_MIN_NEIGHBORHOODS` שכונות - ומתויגות לפי היישוב השכיח שלהן. שום מזהה קהילה אינו מקובע בקוד באף מקום (המקור קיבע קהילה 34 = נתניה ו-23 = בית שאן, מה שנשבר ברגע שזיהוי הקהילות מורץ מחדש).

כל שלושת הקווים הם התאמות OLS על נתונים **לא גזומים**, בהתאמה לערכי ה-rho המצוטטים. תצוגת ציר ה-y נחתכת באחוזון ה-`DISPLAY_TRIM_PCT` כדי שהקווים יישארו על המסך, ו-`tables/trend_fit_sensitivity.csv` מתעד עד כמה כל שיפוע היה משתנה לו הגזימה הזו הייתה מיושמת גם על ההתאמה - אי-ההתאמה שהייתה בגרסה הקודמת.

פלט: `figures/socioeconomic_simpson_paradox.png`.

In [ ]:
example_pool = valid[valid['n_neighborhoods'] >= FIG_MIN_NEIGHBORHOODS].sort_values('rho_use_per_capita')
simpson_picks = [(example_pool.iloc[0], '#2980b9'), (example_pool.iloc[-1], '#e67e22')]

all_nb = nb.dropna(subset=['socio_cluster', 'stop_use_per_1000'])
y_cap = float(np.nanpercentile(all_nb['stop_use_per_1000'], DISPLAY_TRIM_PCT))

fig, ax = plt.subplots(figsize=(12, 7))
xs = np.array([1.0, 10.0])
coef_nat = ols_slope(all_nb['socio_cluster'], all_nb['stop_use_per_1000'])
ax.plot(xs, np.polyval(coef_nat, xs), color='#222222', linewidth=3.5, linestyle='--',
        label='National, all %d neighbourhoods (rho = %+.2f) - the trends cancel out'
              % (NATIONAL_N, NATIONAL_RHO), zorder=5)

sensitivity_rows = []
def record_sensitivity(name, frame):
    full = ols_slope(frame['socio_cluster'], frame['stop_use_per_1000'])
    cap = float(np.nanpercentile(frame['stop_use_per_1000'], DISPLAY_TRIM_PCT))
    trimmed_frame = frame[frame['stop_use_per_1000'] <= cap]
    trimmed = ols_slope(trimmed_frame['socio_cluster'], trimmed_frame['stop_use_per_1000'])
    rho, p_value, n = spearman(frame['socio_cluster'], frame['stop_use_per_1000'])
    sensitivity_rows.append({
        'series': name,
        'n_untrimmed': int(len(frame)),
        'n_after_p%d_trim' % DISPLAY_TRIM_PCT: int(len(trimmed_frame)),
        'ols_slope_untrimmed_used_in_figures': None if full is None else float(full[0]),
        'ols_slope_trimmed_old_behaviour': None if trimmed is None else float(trimmed[0]),
        'spearman_rho_untrimmed': rho,
        'p_value_untrimmed': p_value,
    })

record_sensitivity('NATIONAL (all neighbourhoods)', all_nb)
for row, color in simpson_picks:
    group = nb[nb['community'] == row['community']].dropna(subset=['socio_cluster', 'stop_use_per_1000'])
    shown = group[group['stop_use_per_1000'] <= y_cap]
    ax.scatter(shown['socio_cluster'], shown['stop_use_per_1000'], s=70, alpha=0.55,
               color=color, edgecolor='white', linewidth=0.5, zorder=3)
    coef = ols_slope(group['socio_cluster'], group['stop_use_per_1000'])
    gx = np.array([group['socio_cluster'].min(), group['socio_cluster'].max()])
    ax.plot(gx, np.polyval(coef, gx), color=color, linewidth=4.5, zorder=4,
            label='%s  (rho = %+.2f, p_FDR = %.3g, n = %d)'
                  % (row['community_label'], row['rho_use_per_capita'], row['p_fdr_bh'],
                     int(row['n_neighborhoods'])))
    record_sensitivity(row['community_label'], group)

ax.set_ylim(0, 1.05 * y_cap)
ax.set_xlim(0.5, 10.5)
ax.set_xlabel('Socioeconomic cluster of the neighbourhood  (1 = weakest, 10 = strongest)')
ax.set_ylabel('Service per capita (scheduled stop calls / 1,000 residents)')
ax.set_title("Same country, opposite conclusion in each city - Simpson's paradox",
             fontsize=15, fontweight='bold', pad=12)
ax.legend(loc='upper right', fontsize=11, frameon=True, title='Trend fitted within:')
ax.text(0.02, 0.97,
        'Lines are OLS fits on untrimmed data, matching the quoted rho.\n'
        'Only the y-axis view is clipped (p%d) so every line stays on screen.' % DISPLAY_TRIM_PCT,
        transform=ax.transAxes, ha='left', va='top', fontsize=11, color='#333333',
        bbox=dict(boxstyle='round,pad=0.4', fc='#f4f4f4', ec='#cccccc', lw=1.0))
save_fig(fig, 'socioeconomic_simpson_paradox.png')

trend_sensitivity = pd.DataFrame(sensitivity_rows)
print(trend_sensitivity.to_string(index=False))

## שמירת פלטי השלב

הכול נכתב תחת `outputs/nb/08_socioeconomic_equity/`. הייצוא ברמת התחנה נושא במכוון את `trip_graph_degree` וללא שום עמודה בשם `degree`, כך ששלב במורד הזרם לא יוכל לקלוט בשקט את דרגת גרף הקרבה שהוצאה משימוש, אשר הופיעה בקובץ שפורסם `stops_with_socioeconomic.csv`. הקובץ `socioeconomic_summary.json` מתעד את קו הבסיס הארצי יחד עם ספירות ההשוואות המרובות, כך שניתן לצטט את המספרים המרכזיים ללא הרצה חוזרת של דבר.

In [ ]:
stop_export_cols = ['stop_id', 'stop_name', 'region', 'metro', 'community', 'lat', 'lon',
                    'trip_graph_degree', 'trip_graph_weighted_degree', 'stop_use_count',
                    'is_critical', 'socio_join_method', 'socio_join_distance_m', 'socio_unit_id',
                    'socio_locality', 'socio_locality_en', 'socio_cluster', 'socio_cluster_group',
                    'socio_index_value', 'socio_population']
stop_export_cols = [c for c in stop_export_cols if c in joined.columns]
stop_export = pd.DataFrame(joined[stop_export_cols])

tables = STAGE / 'tables'
stop_export.to_csv(tables / 'stops_with_socioeconomic.csv', index=False, encoding='utf-8-sig')
nb.to_csv(tables / 'socioeconomic_neighborhood_access.csv', index=False, encoding='utf-8-sig')
cluster_summary.to_csv(tables / 'socioeconomic_cluster_summary.csv', index=False, encoding='utf-8-sig')
community_summary.to_csv(tables / 'community_socioeconomic_summary.csv', index=False, encoding='utf-8-sig')
national_correlations.to_csv(tables / 'socioeconomic_national_correlations.csv', index=False, encoding='utf-8-sig')
within.to_csv(tables / 'socioeconomic_within_cluster_correlation.csv', index=False, encoding='utf-8-sig')
trend_sensitivity.to_csv(tables / 'trend_fit_sensitivity.csv', index=False, encoding='utf-8-sig')
(tables / 'socioeconomic_join_quality.json').write_text(
    json.dumps(join_quality, ensure_ascii=False, indent=2), encoding='utf-8')

summary = {
    'sign_convention': 'raw Spearman rho; negative = more service per capita in weaker neighbourhoods',
    'service_variable': 'scheduled stop calls per 1,000 residents, at CBS statistical-area level',
    'degree_source': 'trip-adjacency graph (the retired 500 m proximity degree is NOT used)',
    'national_neighborhood_rho': NATIONAL_RHO,
    'national_neighborhood_p': NATIONAL_P,
    'national_neighborhood_n': NATIONAL_N,
    'community_level_rho': rho_flat,
    'community_level_p': p_flat,
    'community_level_n': n_flat,
    'communities_tested': N_TESTS,
    'within_rho_min': float(valid['rho_use_per_capita'].min()),
    'within_rho_max': float(valid['rho_use_per_capita'].max()),
    'significant_raw': int(within['significant_raw_05'].sum()),
    'significant_fdr_bh': int(within['significant_fdr_05'].sum()),
    'significant_bonferroni': int(within['significant_bonferroni_05'].sum()),
    'expected_false_positives_at_raw_alpha': ALPHA * N_TESTS,
    'cbs_layer': CBS_LAYER_URL,
    'cbs_index_value_available': HAS_INDEX_VALUE,
    'stop_metrics_source': str(STOP_METRICS_PATH),
}
(tables / 'socioeconomic_summary.json').write_text(
    json.dumps(summary, ensure_ascii=False, indent=2, default=float), encoding='utf-8')

print('tables written to', tables)
for path in sorted(tables.iterdir()):
    print('  ', path.name)
print()
print('figures written to', STAGE / 'figures')
for path in sorted((STAGE / 'figures').iterdir()):
    print('  ', path.name)

## מסקנות

1. **ברמה הארצית, המעמד הסוציו-אקונומי כמעט אינו מנבא את השירות התחבורתי לנפש.** מצרף של כל אזור סטטיסטי במדינה נותן Spearman rho של כ--0.15 - מובהק סטטיסטית רק משום שהמדגם מונה אלפי שכונות, וחלש בהרבה מכדי לתאר דבר-מה על מקום מסוים. בצבירה ברמה גבוהה יותר, לקהילות Louvain שלמות, אף זה נעלם: המתאם ברמת הקהילה אינו מובהק.

2. **בתוך קהילה בודדת הקשר לרוב חזק, וסימנו אינו יציב.** על פני כ-50-60 הקהילות הגדולות דיין לבחינה, ה-rho בתוך הקהילה נע בערך בין -0.7 ל-+0.5. חלק מהמטרופולינים מרכזים בבירור שירות לנפש בשכונותיהם החלשות; אחרים עושים את ההפך. מיצוע ארצי שלהם מקזז את שתי ההשפעות - פרדוקס סימפסון, והתוצאה המרכזית של מחברת זו.

3. **הגרסה הכנה של הממצא קטנה מהגרסה הגולמית.** מבין הקהילות שנבחנו, פחות באופן ניכר שורדות את תיקון ה-FDR של Benjamini-Hochberg מאשר עוברות סף p גולמי של 0.05, ופחות מכך שורדות את Bonferroni. עם כ-50 מבחנים בו-זמניים, כשלוש תוצאות "מובהקות" גולמיות צפויות מרעש בלבד. הפרדוקס עצמו יציב - *פיזור הסימנים* גדול בהרבה מרעש הדגימה - אך טענות ברמת עיר בודדת יש לצטט מעמודת ה-FDR ולא מהעמודה הגולמית.

4. **מה שאין בכך כדי להראות.** עצירות מתוכננות לתושב מודדות *היצע*, לא נגישות, לא ביקוש נסיעות ולא זמן נסיעה; שכונה על מסדרון עורקי עמוס מקבלת ציון גבוה גם אם אף קו אינו מגיע לאן שתושביה צריכים. קהילות Louvain הן פרוקסי לערים ולמטרופולינים, לא גבולות מוניציפליים, ולכן תווית "עיר" היא היישוב השכיח של הקהילה ולא עובדה מנהלית. כחמישית מהתחנות משויכות לאזור הסטטיסטי שלהן באמצעות נפילה לאחור לפוליגון הקרוב ביותר ולא באמצעות הכלה מדויקת, וכ-6% מאזורי הלמ"ס אינם נושאים נתון אוכלוסייה ולכן נושרים מכל סטטיסטיקה לנפש.

5. **סיכון שחזור שיש לציין בהגשה.** הנתונים הסוציו-אקונומיים נשלפים בזמן ריצה משירות ArcGIS חיצוני ואינם מצורפים למאגר זה. השכבה כבר השתנתה פעם אחת - `INDEX_VALUE_2021` ו-`CLUSTER_2021` מוגשים כעת כ-`NULL`, כך שרק אשכול `eshkol_mad` בטווח 1-10 שמיש - ומשמעות הדבר היא ששחזור מספרי מדויק של הדוח הקודם אינו מובטח. שמרו את קובץ ה-GeoJSON השמור במטמון ב-`outputs/nb/08_socioeconomic_equity/data/`.